In [16]:
import sys
import os

# Adds the parent/root directory to the python path
sys.path.append(os.path.abspath(os.path.join('..')))

In [17]:
%pwd

'/mnt/D6A6D17EA6D15F93/AI-ML/REAL WORLD PROJECTS/MACHINE LEARNING/Chest-Disease-Identification'

In [18]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list

In [19]:
from src.cnnClassifier.constants import *
from src.cnnClassifier.utils.common import read_yaml, create_directories
import tensorflow as tf

import warnings
warnings.filterwarnings('ignore')

In [20]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

        

    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir, "Chest-CT-Scan-data")
        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

        return training_config

In [21]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [22]:
import tensorflow as tf
from pathlib import Path

class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        # 1. Load the model architecture and weights
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )
        
        # 2. Try fetching from self.config, fallback to a standard default learning rate if missing
        learning_rate = getattr(self.config, 'params_learning_rate', 0.01) 
        
        # 3. Re-compile the model
        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )


    def train_valid_generator(self):
        datagenerator_kwargs = dict(rescale=1./255, validation_split=0.20)
        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(**datagenerator_kwargs)
        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data, subset="validation", shuffle=False, **dataflow_kwargs
        )

        train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            rotation_range=40, horizontal_flip=True, width_shift_range=0.2,
            height_shift_range=0.2, shear_range=0.2, zoom_range=0.2, **datagenerator_kwargs
        ) if self.config.params_is_augmentation else valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data, subset="training", shuffle=True, **dataflow_kwargs
        )
    
    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)
    
    def train(self):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator, epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch, validation_steps=self.validation_steps,
            validation_data=self.valid_generator
        )
        self.save_model(path=self.config.trained_model_path, model=self.model)


In [23]:
import os

# Check if current directory is 'research' and move up one level
if os.getcwd().endswith("research"):
    os.chdir("../")

print(f"Current Working Directory: {os.getcwd()}")


Current Working Directory: /mnt/D6A6D17EA6D15F93/AI-ML/REAL WORLD PROJECTS/MACHINE LEARNING/Chest-Disease-Identification


In [24]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
    
except Exception as e:
    raise e

[2026-10-02 13:49:45,614: INFO: common : Yaml file config/config.yaml loaded successfully]
[2026-10-02 13:49:45,615: INFO: common : Yaml file params.yaml loaded successfully]
[2026-10-02 13:49:45,616: INFO: common : Create directory at artifacts]
[2026-10-02 13:49:45,617: INFO: common : Create directory at artifacts/training]
[2026-10-02 13:49:45,782: WARNING: saving_utils : Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Found 68 images belonging to 2 classes.
Found 275 images belonging to 2 classes.
17/17 ━━━━━━━━━━━━━━━━━━━━ 21s 1s/step - accuracy: 0.7645 - loss: 3.2752 - val_accuracy: 0.9844 - val_loss: 0.0266
[2026-10-02 13:50:07,335: WARNING: saving_api : You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` o